In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Desktop\Capgemini Training\Capgemini Sprint Evaluation\Pollution_Control--CG\dataset\pollution.csv")
df.columns = df.columns.str.strip()

print("Initial Shape:", df.shape)
df.head()

Initial Shape: (6630, 22)


,Record_ID,Plant_ID,Stack_ID,TS,PM2.5,SO2,NOx,Unit,Latitude,Longitude,...,Calibration_Offset,Span_Factor,Lab_Entry_1,Lab_Entry_2,Maintenance_Flag,Inspection_Notes,PM25_Limit,SO2_Limit,NOx_Limit,Correction_Log
0,E20000,PL-08,STK-008-A,2026-01-01 00:00+05:30,73.1,18.7,25.4,ug/m³,13.2214,81.4807,...,0.0,1.0,73.1,73.1,NaN,NaN,NaN,NaN,NaN,NaN
1,E20001,PL-01,STK-001-A,2026-01-01 00:10,73.1,26.2,49.9,ug/m3,12.6208,79.5575,...,0.0,1.0,73.1,73.1,NaN,NaN,NaN,NaN,NaN,NaN
2,E20002,PL-10,STK-010-A,2026-01-01 00:20,83.0,26.1,45.5,UG/m3,12.7270,81.2451,...,0.0,1.0,83.0,83.0,NaN,NaN,NaN,NaN,NaN,NaN
3,E20003,PL-17,STK-017-A,2026-01-01 00:30,64.5,31.0,49.9,ug/m3,13.5187,81.0188,...,0.0,1.0,64.5,64.5,NaN,NaN,NaN,NaN,NaN,NaN
4,E20004,PL-11,STK-011-A,2026-01-01 00:40,82.1,22.0,51.9,µg/m³,13.2038,80.0315,...,0.0,1.0,82.1,82.1,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# 1. DATETIME NORMALIZATION

def fix_datetime(ts):
    ts = str(ts)

    if "24:" in ts:
        date_part = ts.split()[0]
        time_part = ts.split()[1].replace("24:", "00:")
        new_date = pd.to_datetime(date_part, dayfirst=True) + pd.Timedelta(days=1)
        return pd.to_datetime(str(new_date.date()) + " " + time_part)

    return pd.to_datetime(ts, errors="coerce", dayfirst=True)

In [10]:
# 2. UNIT STANDARDIZATION

df["Unit"] = (
    df["Unit"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "µg/m³": "ug/m3",
        "ug/m3 ": "ug/m3",
        "ug/m3": "ug/m3"
    })
)

df["Unit"].unique()

<StringArray>
['ug/m³', 'ug/m3', 'mg/nm3']
Length: 3, dtype: str

In [11]:
# 3. STATUS WHITESPACE + CASING STANDARDIZATION

df["Status"] = (
    df["Status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df["Status"].unique()

<StringArray>
['OK', 'MAINTENANCE', 'MAINT', 'FAULT', 'UNKNOWN', '--', '0', nan, 'ERR', '?']
Length: 10, dtype: str

In [12]:
# 4. COORDINATE VALIDATION

df["Lat"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Lon"] = pd.to_numeric(df["Longitude"], errors="coerce")

df.loc[(df["Lat"] < -90) | (df["Lat"] > 90), "Lat"] = np.nan
df.loc[(df["Lon"] < -180) | (df["Lon"] > 180), "Lon"] = np.nan

df[["Lat","Lon"]].head()

,Lat,Lon
0,13.2214,81.4807
1,12.6208,79.5575
2,12.7270,81.2451
3,13.5187,81.0188
4,13.2038,80.0315


In [13]:
# 4. POLLUTANT SANITY CHECK (NEGATIVE AND IMPOSSIBLE)

pollutants = ["PM2.5", "SO2", "NOx"]

for col in pollutants:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[df[col] < 0, col] = np.nan
    df.loc[df[col] > 5000, col] = np.nan

df[pollutants].describe()

,PM2.5,SO2,NOx
count,6249.000000,6239.000000,6197.000000
mean,73.215490,27.313816,55.429401
std,78.998864,29.449708,76.912167
min,0.000000,0.100000,0.000000
25%,39.900000,18.300000,29.900000
50%,69.700000,25.200000,50.000000
75%,100.000000,32.000000,70.200000
max,2239.500000,582.300000,1808.200000


In [14]:
# 6. DUPLICATE MERGE

df = df.sort_values("TS")

df = (
    df.groupby(["Plant_ID", "TS"], as_index=False)
    .agg({
        "PM2.5": "mean",
        "SO2": "mean",
        "NOx": "mean",
        "Lat": "first",
        "Lon": "first",
        "Status": "first",
        "Unit": "first",
        "Record_ID": "first"
    })
)

print("After Duplicate Merge:", df.shape)

After Duplicate Merge: (6500, 10)


In [15]:
# 7. SENSOR CALIBRATION

df["PM2.5"] *= 0.98
df["SO2"] *= 1.02
df["NOx"] *= 1.01

df[pollutants].head()

,PM2.5,SO2,NOx
0,85.162,25.092,41.713
1,NaN,19.890,9.797
2,68.894,26.418,49.995
3,71.932,26.826,31.613
4,NaN,NaN,42.016


In [16]:
# 8. OUTLIER FILTER

for col in pollutants:
    med = df[col].rolling(5, center=True).median()
    diff = abs(df[col] - med)
    df.loc[diff > 3 * df[col].std(), col] = np.nan

df[pollutants].describe()

,PM2.5,SO2,NOx
count,6104.000000,6085.000000,6048.000000
mean,68.743412,25.890600,52.091009
std,44.758167,13.311747,45.651785
min,0.000000,0.102000,0.000000
25%,39.077500,18.666000,30.199000
50%,68.110000,25.500000,50.298000
75%,97.804000,32.436000,70.599000
max,1937.950000,452.778000,1826.282000


In [17]:
# 9. GAP FILLING

df[pollutants] = df[pollutants].interpolate()

df[pollutants].isna().sum()

PM2.5    0
SO2      0
NOx      0
dtype: int64

In [18]:
# 10. DOWNTIME LABEL

df["Downtime"] = np.where(df["Status"] == "MAINT", 1, 0)

df[["Status","Downtime"]].head()

,Status,Downtime
0,MAINTENANCE,0
1,MAINT,1
2,OK,0
3,MAINT,1
4,MAINTENANCE,0


In [19]:
# 11. LOD HANDLING

LOD = {"PM2.5": 5, "SO2": 3, "NOx": 4}

for col in pollutants:
    df.loc[df[col] < LOD[col], col] = LOD[col] / 2

df[pollutants].describe()

,PM2.5,SO2,NOx
count,6500.000000,6500.000000,6500.000000
mean,68.815031,25.969251,52.454765
std,44.296925,13.554530,46.844441
min,2.500000,1.500000,2.000000
25%,39.690000,18.870000,30.906000
50%,68.012000,25.500000,50.399000
75%,97.020000,32.295750,69.993000
max,1937.950000,452.778000,1826.282000


In [20]:
# 12. STATUS NORMALIZATION

df["Status"] = df["Status"].replace({
    "MAINTENANCE": "MAINT",
    "MAINT ": "MAINT",
    "MAINTENANCE ": "MAINT",
    "OK ": "OK",
    "FAULTY": "FAULT",
    "ERROR": "FAULT"
})

df["Status"].unique()

<StringArray>
['MAINT', 'OK', 'FAULT', 'UNKNOWN', '?', 'ERR', '--', '0', nan]
Length: 9, dtype: str

In [21]:
# 13. STACK TO PLANT MAPPING VALIDATION

# Dummy master mapping table (since Stack_ID not present in raw dataset)
plant_stack_map = {
    "PL-01": "ST-01",
    "PL-02": "ST-02",
    "PL-03": "ST-03",
    "PL-04": "ST-04",
    "PL-05": "ST-05",
    "PL-06": "ST-06",
    "PL-07": "ST-07",
    "PL-08": "ST-08",
    "PL-09": "ST-09"
}

# Create Stack_ID from Plant_ID
df["Stack_ID"] = df["Plant_ID"].map(plant_stack_map)

# Validation flag (1 = invalid mapping, 0 = valid)
df["Stack_Map_Flag"] = df["Stack_ID"].isna().astype(int)

# Optional: Keep only valid rows (comment this if you want audit trace)
# df = df[df["Stack_Map_Flag"] == 0]

# Verification
df[["Plant_ID", "Stack_ID", "Stack_Map_Flag"]].head()

,Plant_ID,Stack_ID,Stack_Map_Flag
0,PL-01,ST-01,0
1,PL-01,ST-01,0
2,PL-01,ST-01,0
3,PL-01,ST-01,0
4,PL-01,ST-01,0


In [22]:
# 14. UNIT CONVERSION

df.loc[df["Unit"] == "mg/nm3", pollutants] *= 1000
df["Unit"] = "ug/m3"

df["Unit"].unique()

<StringArray>
['ug/m3']
Length: 1, dtype: str

In [23]:
# 15. SOURCE TAGGING

df["Source"] = "STACK"

df["Source"].head()

0    STACK
1    STACK
2    STACK
3    STACK
4    STACK
Name: Source, dtype: str

In [24]:
# 16. QC FLAG

df["QC_Flag"] = np.where(df[pollutants].isna().any(axis=1), 1, 0)

df["QC_Flag"].value_counts()

QC_Flag
0    6500
Name: count, dtype: int64

In [25]:
# 17. CONVERSION TO IST

df["TS"] = pd.to_datetime(df["TS"], errors="coerce")

if df["TS"].dt.tz is None:
    df["TS"] = df["TS"].dt.tz_localize("Asia/Kolkata")
else:
    df["TS"] = df["TS"].dt.tz_convert("Asia/Kolkata")

df["TS"].head()

0   2026-01-02 23:00:00+05:30
1                         NaT
2                         NaT
3                         NaT
4   2026-05-02 22:10:00+05:30
Name: TS, dtype: datetime64[us, Asia/Kolkata]

In [26]:
# 18. PII REMOVAL

df["Inspection_Notes"] = "REDACTED"

In [27]:
# 19. THRESHOLD JOIN

threshold = 60
df["PM25_Exceed"] = np.where(df["PM2.5"] > threshold, 1, 0)

df["PM25_Exceed"].value_counts()

PM25_Exceed
1    4646
0    1854
Name: count, dtype: int64

In [28]:
# 20. AUDIT TRAIL

df["Audit"] = "Cleaned_v1"

In [2]:
# FINAL RENAME + SAVE + New File

df = df.rename(columns={
    "PM2.5": "PM2_5_ugm3",
    "SO2": "SO2_ugm3",
    "NOx": "NOx_ugm3"
})

df.to_csv("cleaned_pollution.csv", index=False)
df

,Record_ID,Plant_ID,Stack_ID,TS,PM2_5_ugm3,SO2_ugm3,NOx_ugm3,Unit,Latitude,Longitude,...,Calibration_Offset,Span_Factor,Lab_Entry_1,Lab_Entry_2,Maintenance_Flag,Inspection_Notes,PM25_Limit,SO2_Limit,NOx_Limit,Correction_Log
0,E20000,PL-08,STK-008-A,2026-01-01 00:00+05:30,73.1,18.7,25.4,ug/m³,13.2214,81.4807,...,0.0,1.0,73.1,73.1,NaN,NaN,NaN,NaN,NaN,NaN
1,E20001,PL-01,STK-001-A,2026-01-01 00:10,73.1,26.2,49.9,ug/m3,12.6208,79.5575,...,0.0,1.0,73.1,73.1,NaN,NaN,NaN,NaN,NaN,NaN
2,E20002,PL-10,STK-010-A,2026-01-01 00:20,83.0,26.1,45.5,UG/m3,12.7270,81.2451,...,0.0,1.0,83.0,83.0,NaN,NaN,NaN,NaN,NaN,NaN
3,E20003,PL-17,STK-017-A,2026-01-01 00:30,64.5,31.0,49.9,ug/m3,13.5187,81.0188,...,0.0,1.0,64.5,64.5,NaN,NaN,NaN,NaN,NaN,NaN
4,E20004,PL-11,STK-011-A,2026-01-01 00:40,82.1,22.0,51.9,µg/m³,13.2038,80.0315,...,0.0,1.0,82.1,82.1,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6625,E22690,PL-01,STK-001-A,2026-01-19 16:20,5.7,22.8,24.7,mg/nm3,12.6203,79.9658,...,0.0,1.0,5.7,5.7,NaN,NaN,NaN,NaN,NaN,NaN
6626,E26085,PL-14,STK-014-A,2026-02-12 06:10,129.6,35.8,80.4,UG/m3,13.4076,80.4396,...,0.0,1.0,125.6,125.6,NaN,NaN,NaN,NaN,NaN,NaN
6627,E20397,PL-02,STK-002-A,2026-01-03 18:10,11.0,NaN,11.0,UG/m3,13.6809,79.5160,...,0.0,1.0,10.6,10.6,NaN,NaN,NaN,NaN,NaN,NaN
6628,E24722,PL-02,STK-002-A,2026-02-02 19:00,17.1,9.0,24.2,mg/Nm3,14.4605,80.7726,...,0.0,1.0,18.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN
